# Exercise 9: Partitioned Iceberg and Spark DataFrame insertion

This exercise will create a partitioned iceberg table, which will be populated via a Spark DataFrame, then a snapshot validation will be performed and metadata check

In [0]:
# Step 0: DataFrame Creation

from pyspark.sql import Row
from pyspark.sql import functions as F

data = [
    Row(country="MX", year=2024, value=10),
    Row(country="MX", year=2025, value=20),
    Row(country="US", year=2024, value=30),
    Row(country="US", year=2025, value=40),
]

df = spark.createDataFrame(data)
df.show()

+-------+----+-----+
|country|year|value|
+-------+----+-----+
|     MX|2024|   10|
|     MX|2025|   20|
|     US|2024|   30|
|     US|2025|   40|
+-------+----+-----+



In [0]:
%sql
-- Step 1: Partitioned Iceberg Table Creation
CREATE TABLE IF NOT EXISTS workspace.default.iceberg_partitioned (
  country STRING,
  year INT,
  value INT
)
USING ICEBERG
PARTITIONED BY (country, year);

In [0]:
# Step 2: Data Insertion using the Spark DataFrame
(
    df.write
      .format("iceberg")
      .mode("append")
      .saveAsTable("analytics.raw.iceberg_partitioned")
)

In [0]:
%sql
-- Step 3: Validation
SELECT * FROM workspace.default.iceberg_partitioned;

In [0]:
%sql
-- Step 4: Show Partitions
SHOW PARTITIONS workspace.default.iceberg_partitioned;

-- Step 5: Show Iceberg Metadata
DESCRIBE DETAIL workspace.default.iceberg_partitioned;

-- Step 6: Check Snapshots
CALL workspace.system.snapshots('workspace.default.iceberg_partitioned')